# 🧹 Actividad 05 — Limpieza y Estandarización
Geo: MAYÚSCULAS SIN TILDES | NLP: Regex HTML/URL

In [ ]:
%matplotlib inline
import os, sys, json, re, warnings, unicodedata
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(os.path.abspath('..'))
with open('data/02_interim/pipeline_config.json','r',encoding='utf-8') as f:
    CFG = json.load(f)
DIRS=CFG['DIRS']; INTERIM=DIRS['interim']; REPORTS=DIRS['reports']; PROCESSED=DIRS['processed']
print(f"✅ Setup OK | {os.getcwd()}")

## 5.1 Funciones de limpieza

In [ ]:

def norm_geo(t):
    if not isinstance(t,str): return t
    t=t.strip().upper()
    for a,b in [('Á','A'),('É','E'),('Í','I'),('Ó','O'),('Ú','U')]: t=t.replace(a,b)
    return ''.join(c for c in unicodedata.normalize('NFD',t) if unicodedata.category(c)!='Mn')
def clean_nlp(t):
    if not isinstance(t,str): return ''
    t=re.sub(r'<[^>]+>',' ',t); t=re.sub(r'https?://\S+|www\.\S+',' ',t)
    t=re.sub(r'[^\w\s\.,;:\-\(\)¿\?¡\!áéíóúÁÉÍÓÚñÑ]',' ',t)
    return re.sub(r'\s+',' ',t).strip()
print("Funciones definidas: norm_geo() y clean_nlp()")


## 5.2 MIDAGRI — Antes vs Después

In [ ]:

df_m = pd.read_csv(f"{INTERIM}/midagri_limon_raw.csv")
antes = df_m[['Dpto','Prov','dsc_Cultivo']].head(6).copy()
df_m = df_m.rename(columns={'anho':'anho','mes':'mes','COD_UBIGEO':'cod_ubigeo','Dpto':'departamento',
    'Prov':'provincia','Dist':'distrito','dsc_Cultivo':'cultivo','PRODUCCION(t)':'produccion_t',
    'COSECHA (ha)':'cosecha_ha','MTO_PRECCHAC (S/ x kg)':'precio_chacra_kg'})
for c in ['departamento','provincia','distrito','cultivo']: df_m[c]=df_m[c].apply(norm_geo)
df_m['fecha_evento']=df_m['anho'].astype(str)+'-'+df_m['mes'].astype(str).str.zfill(2)
df_m['produccion_t']=pd.to_numeric(df_m['produccion_t'],errors='coerce').fillna(0)
df_m['cosecha_ha']=pd.to_numeric(df_m['cosecha_ha'],errors='coerce').fillna(0)
df_m['precio_chacra_kg']=pd.to_numeric(df_m['precio_chacra_kg'],errors='coerce')
print("ANTES:"); display(antes)
print("DESPUÉS:"); display(df_m[['departamento','provincia','cultivo']].head(6))
out=f"{INTERIM}/midagri_limon_clean.csv"; df_m.to_csv(out,index=False,encoding='utf-8-sig')
print(f"[OK] {out} — {len(df_m):,} filas")


## 5.3 INDECI — Filtro Fenómenos

In [ ]:

df_ev=pd.read_csv(f"{INTERIM}/indeci_eventos_dbf.csv",low_memory=False)
df_ev=df_ev.rename(columns={c:c for c in df_ev.columns})
if 'departamen' in df_ev.columns: df_ev=df_ev.rename(columns={'departamen':'departamento'})
for c in ['departamento','provincia','fenomeno']:
    if c in df_ev.columns: df_ev[c]=df_ev[c].apply(norm_geo)
df_ev['fecha']=pd.to_datetime(df_ev['fecha'],errors='coerce')
df_ev['fecha_evento']=df_ev['fecha'].dt.strftime('%Y-%m')
for s,d in [('safecta','personas_afectadas'),('sdamni','personas_damnificadas'),
            ('sareaculti','has_cultivo_afectadas'),('sareacul_1','has_cultivo_perdidas')]:
    if s in df_ev.columns: df_ev[d]=pd.to_numeric(df_ev[s],errors='coerce').fillna(0)
PELIGROS=[p.upper() for p in CFG['PELIGROS_VALIDOS']]
df_clean=df_ev[df_ev['fenomeno'].isin(PELIGROS)].copy()
print(f"Originales: {len(df_ev):,} → Filtrados: {len(df_clean):,} ({len(df_clean)/len(df_ev)*100:.1f}%)")
fig,ax=plt.subplots(figsize=(12,4))
df_clean['fenomeno'].value_counts().plot(kind='bar',ax=ax,color='steelblue',edgecolor='black')
ax.set_title('Fenómenos Hidrometeorológicos Válidos tras Filtro',fontsize=12,fontweight='bold')
ax.tick_params(axis='x',rotation=30); plt.tight_layout(); plt.show()
out=f"{INTERIM}/indeci_eventos_clean.csv"; df_clean.to_csv(out,index=False,encoding='utf-8-sig')
print(f"[OK] {out}")


## 5.4 AGRARIA.PE — Limpieza NLP

In [ ]:

df_n=pd.read_csv(f"{INTERIM}/agraria_noticias_raw.csv")
df_n['titular_clean']=df_n['titular'].apply(clean_nlp)
df_n['cuerpo_clean']=df_n['cuerpo_completo'].apply(clean_nlp)
df_n['fecha']=pd.to_datetime(df_n['fecha'],errors='coerce')
df_n['fecha_evento']=df_n['fecha'].dt.strftime('%Y-%m')
orig=df_n['cuerpo_completo'].astype(str).str.len().mean()
clean=df_n['cuerpo_clean'].str.len().mean()
print(f"Longitud media original: {orig:,.0f} | limpia: {clean:,.0f} chars ({(1-clean/orig)*100:.1f}% reducción)")
# Muestra
idx=df_n['cuerpo_completo'].astype(str).str.len().idxmax()
print("\nEjemplo Antes:", str(df_n.loc[idx,'cuerpo_completo'])[:150])
print("Ejemplo Después:", df_n.loc[idx,'cuerpo_clean'][:150])
out=f"{INTERIM}/agraria_noticias_clean.csv"; df_n.to_csv(out,index=False,encoding='utf-8-sig')
print(f"\n[OK] {out} — {len(df_n)} noticias")
print("✅ [ACTIVIDAD 05] COMPLETADA")


## 5.5 Estandarización Data Climática (NASA POWER)
Al igual que las fuentes locales, los datos de la NASA se estandarizan para asegurar que los nombres de Departamentos y Provincias coincidan exactamente (MAYÚSCULAS y sin tildes) y que el formato de fecha sea `YYYY-MM`.

In [ ]:

nasa_raw_path = "data/02_interim_nasa/nasa_long_raw.csv"
if os.path.exists(nasa_raw_path):
    df_nasa = pd.read_csv(nasa_raw_path)
    # Estandarización Geo
    for c in ['DEPARTAMENTO', 'PROVINCIA']:
        df_nasa[c] = df_nasa[c].apply(norm_geo)
    
    # Estandarización Temporal
    if 'DATE' in df_nasa.columns:
        df_nasa['fecha_evento'] = pd.to_datetime(df_nasa['DATE']).dt.strftime('%Y-%m')
    
    print(f"Data NASA estandarizada: {len(df_nasa):,} registros")
    display(df_nasa[['DEPARTAMENTO', 'PROVINCIA', 'fecha_evento']].head(3))
else:
    print("Nota: El archivo raw de NASA se procesa en el pipeline especializado 'main_nasa_pipeline.py'")
